In [ ]:
import os
import ctypes

local_lib_path = "./x86_64-linux-gnu"

# 按依赖顺序，提前把缺少的动态库加载到内存中，并设置为全局可见 (RTLD_GLOBAL)
libraries_to_load = [
    "libGLdispatch.so.0", # libGL 的底层依赖 (来自 libglvnd0)
    "libEGL.so.1",
    "libGLX.so.0",        # (来自 libglx0)
    "libGL.so.1",         # (来自 libgl1)
    "libGLESv2.so.2"      # (来自 libgles2)
]

print("开始注入动态库...")
for lib_name in libraries_to_load:
    lib_path = os.path.join(local_lib_path, lib_name)
    if os.path.exists(lib_path):
        try:
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
            print(f"✅ 成功注入: {lib_name}")
        except Exception as e:
            print(f"⚠️ 注入 {lib_name} 失败: {e}")
    else:
        print(f"❌ 找不到文件: {lib_path} (请确认路径对不对)")

print("-" * 40)

try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    print("🎉 MediaPipe!")
except Exception as e:
    print("❌ 导入失败，报错信息：", e)

开始注入动态库...
✅ 成功注入: libGLdispatch.so.0
✅ 成功注入: libEGL.so.1
✅ 成功注入: libGLX.so.0
✅ 成功注入: libGL.so.1
✅ 成功注入: libGLESv2.so.2
----------------------------------------
🎉 MediaPipe!。


In [2]:
import cv2
import numpy as np
from tqdm.auto import tqdm
import json


In [ ]:
JSON_FILE  = ""
VIDEO_ROOT = ""
SAVE_ROOT  = ""
MODEL_PATH = ""
NUM_FRAMES = 32

os.makedirs(SAVE_ROOT, exist_ok=True)


In [1]:
with open(JSON_FILE) as f:
    full_data = json.load(f)

video_ids = [
    vid_id for vid_id, info in full_data.items()
    if os.path.exists(os.path.join(VIDEO_ROOT, f"{vid_id}.mp4"))
]

print(f"Total videos to process: {len(video_ids)}")

# ── 初始化 MediaPipe ──
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3,
    running_mode=vision.RunningMode.IMAGE
)
detector = vision.HandLandmarker.create_from_options(options)

detected     = 0
total_frames = 0
skipped      = 0

for vid_id in tqdm(video_ids, desc="Extracting keypoints"):
    save_path = os.path.join(SAVE_ROOT, f"{vid_id}.npy")
    if os.path.exists(save_path):
        skipped += 1
        continue

    video_path = os.path.join(VIDEO_ROOT, f"{vid_id}.mp4")
    cap   = cv2.VideoCapture(video_path)
    total = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 1)
    indices = set(np.linspace(0, total - 1, NUM_FRAMES, dtype=int))

    keypoints    = []
    last_good    = np.zeros(126, dtype=np.float32)
    vid_detected = 0

    for i in range(total):
        ret, frame = cap.read()
        if not ret:
            break
        if i not in indices:
            continue

        rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        results  = detector.detect(mp_image)

        kp = np.zeros(126, dtype=np.float32)
        if results.hand_landmarks:
            for hand_idx, hand_lm in enumerate(results.hand_landmarks[:2]):
                offset = hand_idx * 63
                for j, lm in enumerate(hand_lm):
                    kp[offset + j*3]     = lm.x
                    kp[offset + j*3 + 1] = lm.y
                    kp[offset + j*3 + 2] = lm.z
            last_good     = kp.copy()
            vid_detected += 1
        else:
            kp = last_good.copy()

        keypoints.append(kp)
        if len(keypoints) == NUM_FRAMES:
            break

    cap.release()

    if not keypoints:
        keypoints = [np.zeros(126, dtype=np.float32)] * NUM_FRAMES
    while len(keypoints) < NUM_FRAMES:
        keypoints.append(keypoints[-1])

    np.save(save_path, np.stack(keypoints))  # [32, 126]

    detected     += vid_detected
    total_frames += len(keypoints)
    detect_rate   = 100. * detected / total_frames if total_frames > 0 else 0

    tqdm.write(f"[{vid_id}] hand detected: {vid_detected}/{len(keypoints)} frames "
               f"| overall detect rate: {detect_rate:.1f}%")

detector.close()

print("\n" + "="*50)
print(f"Done!")
print(f"Processed  : {len(video_ids) - skipped} videos")
print(f"Skipped    : {skipped} videos (already existed)")
print(f"Detect rate: {100.*detected/total_frames:.1f}% "
      f"({detected}/{total_frames} frames)")

NameError: name 'JSON_FILE' is not defined